In [0]:
from pyspark.sql.functions import *

catalog = "bike_data"
silver_schema = "silver"
gold_schema = "gold"

print("=" * 80)
print("GOLD TRANSFORMATION: dim_customers")
print("=" * 80)

# Section 1: Read Silver tables
print("\nSection 1: Reading Silver tables")
customers_df = spark.table(f"{catalog}.{silver_schema}.customers")
erp_customers_df = spark.table(f"{catalog}.{silver_schema}.erp_customers")
locations_df = spark.table(f"{catalog}.{silver_schema}.locations")

print(f"  customers: {customers_df.count():,} rows")
print(f"  erp_customers: {erp_customers_df.count():,} rows")
print(f"  locations: {locations_df.count():,} rows")

# Section 2: Build dim_customers
print("\nSection 2: Building dim_customers")

# Start with CRM customers
dim_customers = customers_df.select(
    col("cst_id"),
    col("cst_firstname"),
    col("cst_lastname"),
    col("cst_marital_status"),
    col("cst_gndr"),
    col("cst_create_date")
)

print(f"  Starting with customers: {dim_customers.count():,}")

# Left join with ERP customers to add birth date
# Join on string: convert cst_id to string
print("  Joining with erp_customers for BDATE...")
erp_cust_clean = erp_customers_df.select(
    col("CID"),
    col("BDATE"),
    col("GEN")
)

dim_customers = dim_customers.join(
    erp_cust_clean,
    dim_customers.cst_id.cast("string") == erp_cust_clean.CID,
    "left"
).drop("CID")

print(f"  After ERP join: {dim_customers.count():,}")

# Left join with locations to add country
print("  Joining with locations for CNTRY...")
loc_clean = locations_df.select(
    col("CID"),
    col("CNTRY")
)

dim_customers = dim_customers.join(
    loc_clean,
    dim_customers.cst_id.cast("string") == loc_clean.CID,
    "left"
).drop("CID")

print(f"  After locations join: {dim_customers.count():,}")

# Section 3: Sanity checks
print("\nSection 3: Sanity checks")

print("  Null values:")
null_check = dim_customers.select([count(when(col(c).isNull(), c)).alias(c) for c in dim_customers.columns])
display(null_check)

print("\n  Data sample:")
display(dim_customers.limit(3))

print("\n  Schema:")
dim_customers.printSchema()

# Section 4: Write to Gold
print("\nSection 4: Writing to Gold table")
gold_table = f"{catalog}.{gold_schema}.dim_customers"
dim_customers.write.mode("overwrite").format("delta").saveAsTable(gold_table)

final_count = dim_customers.count()
print(f"\nWritten to: {gold_table}")
print(f"Row count: {final_count:,}")